# Automatyczne ramki: porownanie 63 linii
GPU, potem Uruchom wszystko. Dane pobieraja sie automatycznie z GitHuba. Nic nie dodawaj recznie. Ostatnia komorka pobiera ZIP. Przy pracy w starej sesji po instalacji zrestartuj sesje i wybierz Uruchom wszystko.
Kazdy model odczyta oryginalne 63 wycinki i wariant 31 nowych ramek + 32 oryginalnych fallbackow. Referencje i historyczna pisownia pozostaja bez zmian. Osobne wyniki dla obu wariantow; to eksperyment deweloperski, nie trening ani dowod SOTA. Nowe ramki nie sa recznie zatwierdzone.
Zrodlo: IMPACT/PSNC, PiotrSty/impact-psnc-polish-ocr, CC-BY-3.0. Zmiany: wycinki PNG i robocze transkrypcje; teraz zmieniono tylko geometrie.


In [ ]:
%pip install -q --upgrade transformers==4.57.6 jiwer==4.0.0 huggingface_hub==0.36.0 sentencepiece==0.2.1

In [ ]:
"""Diagnostic only: user-corrected draft references, not approved benchmark labels."""
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
import base64
import gc
import hashlib
import importlib.metadata
import io
import json
import unicodedata
import zipfile

MODELS = {
    'microsoft/trocr-base-printed': '93450be3f1ed40a930690d951ef3932687cc1892',
    'PiotrSty/trocr-pl-mixed-v3': '85d0c91c26f8e088849096dded7c9ba10b4cd9c9',
}
FROZEN = {'NA2_FT', 'Nowiny_z_Rakuz_FT', 'Powodzenia_FT'}


def digest(data):
    return hashlib.sha256(data).hexdigest()


def pack(manifest, target):
    manifest, target = Path(manifest), Path(target)
    if target.exists():
        raise FileExistsError(target)
    source = [json.loads(s) for s in manifest.read_text(encoding='utf-8').splitlines() if s.strip()]
    rows, images = [], {}
    for i, row in enumerate(source):
        path = (manifest.parent / row['image']).resolve()
        if not path.is_relative_to(manifest.parent.resolve()):
            raise ValueError('Image outside draft directory')
        data = path.read_bytes()
        if digest(data) != row['sha256'] or row['eligible_for_evaluation'] is not False:
            raise ValueError('Expected checksum-verified draft')
        name = f'images/{i:04d}.png'
        images[name] = data
        rows.append({k: row[k] for k in ['id', 'text', 'original_text', 'collection', 'page_id',
                                        'sha256', 'source_review_decision', 'review_status', 'eligible_for_evaluation']})
        rows[-1]['image'] = name
    provenance = {'scope': 'diagnostic-only', 'source_manifest_sha256': digest(manifest.read_bytes()),
                  'reference_status': 'User-corrected draft; repeat review explicitly skipped.',
                  'sampling': '63 count-matched line proposals from 11 of 19 selected regions. Eight regions failed line-count matching. Not full-page evaluation.',
                  'limitations': 'Geometry and Unicode issues remain; no independent double review, near-duplicate audit or proof of upstream training separation.'}
    with zipfile.ZipFile(target, 'x', zipfile.ZIP_DEFLATED) as archive:
        archive.writestr('manifest.json', json.dumps(rows, ensure_ascii=False))
        archive.writestr('provenance.json', json.dumps(provenance))
        for name, data in images.items():
            archive.writestr(name, data)
    return digest(target.read_bytes())


def load_input(data, expected_sha256):
    if digest(data) != expected_sha256:
        raise ValueError('Input ZIP checksum mismatch')
    with zipfile.ZipFile(io.BytesIO(data)) as archive:
        names = archive.namelist()
        if len(names) != len(set(names)):
            raise ValueError('Duplicate ZIP members')
        if sum(i.file_size for i in archive.infolist()) > 50_000_000:
            raise ValueError('Input too large')
        for name in names:
            if PurePosixPath(name).is_absolute() or '..' in PurePosixPath(name).parts or '\\' in name or ':' in name:
                raise ValueError('Unsafe ZIP member')
        rows = json.loads(archive.read('manifest.json'))
        provenance = json.loads(archive.read('provenance.json'))
        if provenance['scope'] != 'diagnostic-only' or not rows or len(rows) > 200:
            raise ValueError('Invalid diagnostic input')
        if len({r['id'] for r in rows}) != len(rows):
            raise ValueError('Duplicate line ID')
        images = []
        for r in rows:
            if r['collection'] in FROZEN or r['eligible_for_evaluation'] is not False:
                raise ValueError('Expected non-test draft records')
            if r['source_review_decision'] not in {'verified', 'proposed', 'needs-review'}:
                raise ValueError('Missing source review decision')
            if not r['text'].strip():
                raise ValueError('Empty reference requires explicit policy')
            image = archive.read(r['image'])
            if digest(image) != r['sha256']:
                raise ValueError('Image checksum mismatch')
            images.append(image)
    return rows, images, provenance


def metrics(rows, predictions):
    from jiwer import cer, wer
    if [r['id'] for r in rows] != [p['id'] for p in predictions]:
        raise ValueError('Reference/prediction ID mismatch')
    normalize = lambda t: ' '.join(unicodedata.normalize('NFC', t).split())
    result = {}
    subsets = {'all_draft_lines': list(range(len(rows))),
               'without_needs_review': [i for i, r in enumerate(rows) if r['source_review_decision'] != 'needs-review']}
    for label, indices in subsets.items():
        if not indices:
            result[label] = {'lines': 0, 'cer': None, 'wer': None}
            continue
        refs = [normalize(rows[i]['text']) for i in indices]
        hyps = [normalize(predictions[i]['text']) for i in indices]
        result[label] = {'lines': len(indices), 'cer': cer(refs, hyps), 'wer': wer(refs, hyps),
                         'lowercase_cer_diagnostic': cer([s.lower() for s in refs], [s.lower() for s in hyps]),
                         'errors': sum(predictions[i]['status'] != 'ok' for i in indices),
                         'empty': sum(not s for s in hyps)}
    return result


def run(data, expected_sha256, runner_sha256=None):
    rows, image_bytes, provenance = load_input(data, expected_sha256)
    import torch
    from PIL import Image
    from transformers import TrOCRProcessor, VisionEncoderDecoderModel
    if not torch.cuda.is_available():
        raise RuntimeError('Select GPU runtime in Colab, then Run all')
    torch.manual_seed(0)
    output = Path('/content') / ('body-dev-diagnostic-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    output.mkdir(parents=True)
    def save(name, value):
        (output / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding='utf-8')
    save('input-manifest.json', rows)
    save('provenance.json', provenance)
    report = {'scope': 'DRAFT diagnostic, not benchmark or SOTA evidence', 'input_zip_sha256': expected_sha256,
              'runner_sha256': runner_sha256,
              'models': MODELS, 'gpu': torch.cuda.get_device_name(0),
              'environment': {p: importlib.metadata.version(p) for p in ['torch', 'transformers', 'Pillow', 'jiwer', 'huggingface_hub']},
              'generation': {'do_sample': False, 'num_beams': 1, 'max_new_tokens': 256, 'dtype': 'float32'},
              'normalization': 'NFC and whitespace; additional lowercase CER reported separately',
              'uncertain_ids': [r['id'] for r in rows if r['source_review_decision'] == 'needs-review'],
              'reference_flags': {r['id']: sum(c == '\ufffd' or unicodedata.category(c) == 'Co' for c in r['text']) for r in rows},
              'results': {}}
    for model_id, revision in MODELS.items():
        model = None
        predictions = []
        try:
            processor = TrOCRProcessor.from_pretrained(model_id, revision=revision, trust_remote_code=False)
            model = VisionEncoderDecoderModel.from_pretrained(model_id, revision=revision, trust_remote_code=False).float().cuda().eval()
            eos = model.generation_config.eos_token_id
            eos = set(eos if isinstance(eos, list) else [eos])
            for row, content in zip(rows, image_bytes):
                pred = {'id': row['id'], 'text': '', 'status': 'error'}
                try:
                    with Image.open(io.BytesIO(content)) as im:
                        pixels = processor(images=im.convert('RGB'), return_tensors='pt').pixel_values.cuda()
                    with torch.inference_mode():
                        ids = model.generate(pixels, do_sample=False, num_beams=1, max_new_tokens=256)[0].tolist()
                    pred.update(text=processor.batch_decode([ids], skip_special_tokens=True)[0], status='ok',
                                token_ids=ids, ended_with_eos=ids[-1] in eos,
                                possibly_truncated=len(ids) >= 257 and ids[-1] not in eos)
                except Exception as exc:
                    pred['error'] = type(exc).__name__ + ': ' + str(exc)
                predictions.append(pred)
        except Exception as exc:
            predictions = [{'id': r['id'], 'text': '', 'status': 'error', 'error': type(exc).__name__ + ': ' + str(exc)} for r in rows]
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
        save(model_id.replace('/', '--') + '.json', predictions)
        report['results'][model_id] = metrics(rows, predictions)
        save('report.json', report)
        print(model_id, json.dumps(report['results'][model_id], indent=2))
    save('checksums.json', {p.name: digest(p.read_bytes()) for p in output.iterdir() if p.is_file()})
    archive = output.with_suffix('.zip')
    with zipfile.ZipFile(archive, 'x', zipfile.ZIP_DEFLATED) as z:
        for path in output.iterdir():
            z.write(path, path.name)
    from IPython.display import HTML, display
    encoded = base64.b64encode(archive.read_bytes()).decode('ascii')
    display(HTML('<a download="' + archive.name + '" href="data:application/zip;base64,' + encoded + '">Pobierz ZIP wynikow</a>'))
    print('Output:', archive)

    return archive

"""Pair frozen original inputs and experimental geometry without dropping IDs."""
import io
import json
import zipfile


def pair_inputs(original_data, original_hash, auto_data, auto_hash, loader):
    original, images_a, _ = loader(original_data, original_hash)
    modified, images_b, provenance = loader(auto_data, auto_hash)
    if [r['id'] for r in original] != [r['id'] for r in modified]:
        raise ValueError('Original/automatic IDs differ')
    for a, b, old_image, new_image in zip(original, modified, images_a, images_b):
        for field in ['text', 'original_text', 'source_review_decision', 'collection', 'page_id']:
            if a[field] != b[field]:
                raise ValueError('Reference or sample metadata changed: ' + field)
        if b['geometry_status'] == 'fallback-original' and old_image != new_image:
            raise ValueError('Fallback image differs from original')
    rows = []
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as z:
        for variant, records, images in [('original', original, images_a), ('automatic_with_fallback', modified, images_b)]:
            for i, (r, image) in enumerate(zip(records, images)):
                path = f'{variant}/{i:04d}.png'
                info = zipfile.ZipInfo(path)
                info.compress_type = zipfile.ZIP_DEFLATED
                z.writestr(info, image)
                rows.append({**r, 'id': r['id'] + '__' + variant, 'comparison_id': r['id'],
                             'geometry_variant': variant, 'image': path,
                             'comparison_geometry_status': modified[i]['geometry_status']})
        z.writestr(zipfile.ZipInfo('manifest.json'), json.dumps(rows, ensure_ascii=False))
        z.writestr(zipfile.ZipInfo('provenance.json'), json.dumps({
            **provenance, 'experiment': 'paired image-only geometry development control',
            'source_archive_hashes': [original_hash, auto_hash],
            'reference_changed': False, 'metrics_scope': 'Report each 63-line arm separately; no pooled score.'}))
    return buffer.getvalue()


def paired_metrics(rows, predictions, base_metrics):
    if [r['id'] for r in rows] != [p['id'] for p in predictions]:
        raise ValueError('Prediction/reference ID mismatch')
    result = {}
    for variant in ['original', 'automatic_with_fallback']:
        indices = [i for i, r in enumerate(rows) if r['geometry_variant'] == variant]
        result[variant] = base_metrics([rows[i] for i in indices], [predictions[i] for i in indices])
        changed = [i for i in indices if rows[i]['comparison_geometry_status'] == 'auto-proposal']
        result[variant]['changed_geometry_only'] = base_metrics(
            [rows[i] for i in changed], [predictions[i] for i in changed])['all_draft_lines']
    return result

"""Reject missing, drifted or stale notebook dependencies before inference."""
import importlib.metadata
import sys

REQUIRED_PACKAGES = {
    'transformers': '4.57.6', 'huggingface_hub': '0.36.0',
    'jiwer': '4.0.0', 'sentencepiece': '0.2.1',
}


def check_colab_environment():
    problems = []
    for name, expected in REQUIRED_PACKAGES.items():
        try:
            installed = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            installed = 'missing'
        if installed != expected:
            problems.append(f'{name}: installed={installed}, required={expected}')
    for name in [*REQUIRED_PACKAGES, 'tokenizers']:
        loaded = sys.modules.get(name)
        version = getattr(loaded, '__version__', None)
        if version is not None:
            try:
                installed = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                installed = 'missing'
            if version != installed:
                problems.append(f'{name}: loaded={version}, installed={installed}; restart required')
    if problems:
        raise RuntimeError('Environment check failed. Run the install cell, restart the Colab session, '
                           'then Run all.\n' + '\n'.join(problems))
    print('Pinned dependencies verified; no stale package versions detected.')

check_colab_environment()
from urllib.request import urlopen
ROOT_URL = 'https://raw.githubusercontent.com/PiotrStyla/OCR_engine/95786263807ff1d640db5c2fc3a16b64ab4c0e3d/'
with urlopen(ROOT_URL + 'experiments/2026-09-21/body-dev-diagnostic/body-dev-input.zip', timeout=120) as response:
    old_data = response.read()
with urlopen(ROOT_URL + 'experiments/2026-09-24/body-auto-geometry/body-auto-geometry-input.zip', timeout=120) as response:
    new_data = response.read()
payload = pair_inputs(old_data, '913ded2bd5d742099567a2652460baaf3c4a8bb16625492d0931607396dbdd55', new_data, 'de90e9b90eb658341e01a16a24e2eb101de4c7cfb50ac1bdee7724878090f278', load_input)
_base_metrics = metrics
def metrics(rows, predictions):
    return paired_metrics(rows, predictions, _base_metrics)
result_archive = run(payload, digest(payload), 'ec4ebdc16f2c24e085cbd2e18fc64ece339464a52497e4dbf9b94b4e906241fc')
with zipfile.ZipFile(result_archive) as evidence:
    final_report = json.loads(evidence.read('report.json'))
failed = []
for model, arms in final_report['results'].items():
    for variant, scores in arms.items():
        score = scores['all_draft_lines']
        if score['errors']:
            failed.append(f"{model} / {variant}: {score['errors']} errors")
        else:
            print(f"{model} / {variant}: CER={score['cer']:.2%}, WER={score['wer']:.2%}")
if failed:
    print('INCOMPLETE RUN. Error-derived CER is not model quality. Download the diagnostic ZIP.\n' + '\n'.join(failed))
else:
    print('COMPLETE: both models finished both arms. Draft development result, not a benchmark.')


In [ ]:
from google.colab import files
files.download(str(result_archive))
